In [ ]:
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

from ontologx.backend import LLMFactory
from ontologx.metrics.ttp_metrics import SessionTacticsMetrics, TacticsPredictor

NEO4J_URL = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "password"  # noqa: S105
TESTS_FILE = "../resources/data/polito/test.ttl"

load_dotenv("../.env")

store = Neo4jGraph(
    url=NEO4J_URL,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
)

In [ ]:
import neo4j
from langchain_core.documents import Document

from ontologx.store import GraphDocument, Node, Relationship
from ontologx.store.neo4j.utils import normalize_output_graph


def get_subgraph_from_node(node_uri: str) -> GraphDocument:
    """Get the subgraph of a node in the store.

    The subgraph will contain all the nodes and relationships connected to the given node, even indirectly.

    Args:
        node_uri (str): The URI of the node to get the subgraph from.

    Returns:
        GraphDocument: The subgraph of the node, with nodes and relationships.

    Raises:
        ValueError: If no subgraph is found for the given node URI.

    """
    # Ugly but quite efficient.
    nodes_subgraphs = store.query(
        """
        MATCH (n {uri: $node_uri})
        CALL apoc.path.subgraphAll(n,
            {labelFilter: '-mlsx__OutputDataset|-mlsx__ExampleDataset|-mlsx__TestDataset'})
        YIELD nodes, relationships
        RETURN
        [node IN nodes | {
        uri: node.uri,
        type: HEAD([label IN LABELS(node) WHERE label <> 'Resource']),
        properties: PROPERTIES(node)
        }] AS nodes,
        [rel IN relationships | {
        source: STARTNODE(rel).uri,
        target: ENDNODE(rel).uri,
        type: TYPE(rel)
        }] AS relationships
        """,
        params={"node_uri": node_uri},
    )

    if not nodes_subgraphs:
        msg = f"No subgraph found for node with URI: {node_uri}"
        raise ValueError(msg)

    nodes_subgraph = nodes_subgraphs[0]

    # Remove the DatasetRow node, as it is not needed in the output graph.
    # However it contains the event message, so it is used as the source of the graph.
    dataset_row_node = next(node for node in nodes_subgraph["nodes"] if node["type"] == "mlsx__DatasetRow")
    nodes_subgraph["nodes"].remove(dataset_row_node)
    nodes_subgraph["relationships"] = [
        relationship
        for relationship in nodes_subgraph["relationships"]
        if relationship["source"] != dataset_row_node["uri"] and relationship["target"] != dataset_row_node["uri"]
    ]

    # The neo4j date and time objects are quite problematic, as they are not JSON serializable.
    # This is a workaround to convert them to strings.
    for node in nodes_subgraph["nodes"]:
        for key, value in node["properties"].items():
            if isinstance(value, neo4j.time.DateTime | neo4j.time.Date | neo4j.time.Time):
                node["properties"][key] = value.iso_format()

    def get_node_id(uri: str) -> str:
        """Get the node id from the node uri."""
        final_str = uri.split("/")[-1]

        if "#" in final_str:
            return final_str.split("#")[-1]

        return final_str

    nodes_dict = {
        node["uri"]: Node(id=get_node_id(node["uri"]), type=node["type"], properties=node["properties"])
        for node in nodes_subgraph["nodes"]
    }

    relationships = (
        [
            Relationship(
                source=nodes_dict[relationship["source"]],
                target=nodes_dict[relationship["target"]],
                type=relationship["type"],
            )
            for relationship in nodes_subgraph["relationships"]
        ]
        if "relationships" in nodes_subgraph
        else []  # The node may not have any relationships
    )

    # Create the context from the event source, if present.
    source_node = next((node for node in nodes_dict.values() if node.type == "olx__Source"), None)
    context = {key: value for key, value in source_node.properties.items() if key != "uri"} if source_node else {}

    return GraphDocument(
        nodes=list(nodes_dict.values()),
        relationships=relationships,
        source=Document(
            page_content=dataset_row_node["properties"]["mlsx__eventMessage"],
            metadata={"context": context},
        ),
    )


def tests() -> list[GraphDocument]:
    """Return a list of test documents from the dataset."""
    test_nodes = store.query(
        """
        MATCH (d:mlsx__TestDataset)-[:mlsx__hasPart]->(r:mlsx__DatasetRow)-[:mlsx__hasLabel]->(e:olx__Event)
        RETURN r.mlsx__eventMessage as eventMessage, e.uri as uri, r.mlsx__hasTactic as tactics
        ORDER BY e.uri
        """,
    )
    tests = []
    for test in test_nodes:
        graph = get_subgraph_from_node(test["uri"])

        source_node = next((node for node in graph.nodes if node.type == "olx__Source"), None)
        context = {}
        if source_node:
            if source_node.properties.get("olx__sourceName"):
                context["sourceName"] = source_node.properties["olx__sourceName"]
            if source_node.properties.get("olx__sourceDevice"):
                context["sourceDevice"] = source_node.properties["olx__sourceDevice"]

        graph.source = Document(
            page_content=test["eventMessage"],
            metadata={"context": context, "tactics": test["tactics"]},
        )
        tests.append(normalize_output_graph(graph))

    return tests

In [ ]:
y_true = tests()
y_pred: list[GraphDocument] = []

for graph in y_true:
    event_node = next((node for node in graph.nodes if node.type == "olx:Event"), None)
    if event_node is None:
        msg = "No Event node found in the graph"
        raise ValueError(msg)

    pred_event_node_uri = store.query(
        """
        MATCH (r:mlsx__Run)-[:mlsx__hasOutput]->(d:mlsx__OutputDataset)
        -[:mlsx__hasPart]->(dr:mlsx__DatasetRow)-[:mlsx__hasLabel]->(e:olx__Event)
        WHERE e.mlsx__eventMessage = $event_message
        RETURN e.uri as uri
        """,
        params={
            "event_message": graph.source.page_content,
        },
    )

    pred_graph = get_subgraph_from_node(pred_event_node_uri[0]["uri"])
    y_pred.append(pred_graph)

In [ ]:
from pathlib import Path
from typing import cast

import pandas as pd

from ontologx.metrics.ttp_metrics import MITRETactic

llm = LLMFactory.create(backend="bedrock", model="claude 3 sonnet", temperature=0)
predictor = TacticsPredictor(
    llm=llm,
    prompt_predict_tactics=Path("../resources/prompts/tactics.system.md").read_text(),
)

sessions_dict: dict[str, list[GraphDocument]] = {}
tactics_dict: dict[str, list[MITRETactic]] = {}
for pred, true in zip(y_pred, y_true, strict=True):
    event_node = next(e for e in pred.nodes if e.type == "olx:Event")
    session_id = cast("str | None", event_node.properties.get("olx:eventSessionID"))

    if session_id is None:
        continue

    if session_id not in sessions_dict:
        sessions_dict[session_id] = []
        tactics_dict[session_id] = []

    sessions_dict[session_id].append(pred)
    true_tactics = [MITRETactic(i) for i in true.source.metadata["tactics"]]
    tactics_dict[session_id] = list(set(tactics_dict[session_id] + true_tactics))

results = []

for session_id, y_pred_session in sessions_dict.items():
    y_true_tactics = tactics_dict[session_id]
    y_pred_tactics = predictor.predict_tactics(y_pred_session)

    metrics = SessionTacticsMetrics(y_labels_pred=y_pred_tactics, y_labels_true=y_true_tactics)
    keys = metrics.precisions.keys()
    metrics_results = {}
    for tactic in keys:
        metrics_results[f"{tactic.name.lower()}_precision"] = metrics.precisions[tactic]
        metrics_results[f"{tactic.name.lower()}_recall"] = metrics.recalls[tactic]
        metrics_results[f"{tactic.name.lower()}_f1_score"] = metrics.f1_scores[tactic]

    results.append(
        {
            "session_id": session_id,
            "pred_tactics": y_pred_tactics,
            "true_tactics": y_true_tactics,
            "metrics": metrics_results,
        },
    )

results_df = pd.DataFrame(results)
results_df.to_csv("tactics_results.csv", index=False)